# PhysAR-IDS -- NB10 (v6): Master results and provenance audit

**What this notebook does.** It trains nothing. It reads every v6 output the earlier notebooks wrote (CSV tables, run manifests, figures) and produces the three things the dissertation needs from them:

1. **Provenance audit:** which notebooks have run, whether every one used the same seeds `[42, 7, 123, 2024, 99]`, pipeline version `v6`, and compatible library versions; which expected files are missing; stale files from older versions.
2. **Master workbook** `physar_master_results_v6.xlsx`: one sheet per results table (plus an index sheet), so every number in the text can be traced to a sheet. Also `master_tables_v6.md` (all tables as Markdown) and one LaTeX `booktabs` file per table in `master_latex_v6/`.
3. **Integrity checks on the tables** (each flag is `CHECK` = look at it, or `INFO` = expected pattern that is listed so nothing is silent): NaN/inf cells, percentage-formatted strings (the 1.58% vs 0.0158 mix-up), `+/-` strings that hide a std of 0.0000 (a deterministic model), suspicious p-values of exactly 0, and fraction-vs-percent columns.
Plus a **figure book** `master_figures_v6.pdf` containing every v6 figure, one per page, so stale v4 figures are easy to spot against it.

**It makes no assumption about column names**, so it keeps working if a table is changed. Set `RESULTS_DIRS` to the folders the notebooks wrote into if auto-discovery fails (NB1-NB8 write to the SmartGrid_Dissertation folder; NB9 writes next to `msu_combined.csv`). The notebook looks in the current folder, its parent, the usual Desktop locations, then walks a few levels below your home folder, and **stops with an error instead of writing empty files** if it finds no v6 outputs.

## 0. Setup

In [ ]:
import os, re, glob, json, time, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg') if os.environ.get('NB10_HEADLESS') else None
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
warnings.filterwarnings('ignore')
PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
RESULTS_DIRS = []          # optional: put folders here to force them, e.g. [r'D:\physar\results']
def _has_v6(d): return os.path.isdir(d) and any(f.endswith(f'_{PIPELINE_VERSION}.csv') or f.endswith(f'_{PIPELINE_VERSION}.json') for f in os.listdir(d))
if not RESULTS_DIRS:
    home = os.path.expanduser('~'); cands = [os.environ.get('PHYSAR_RESULTS', ''), os.getcwd(), os.path.dirname(os.getcwd()),
        r'C:\Users\user\Desktop\smartrid1\SmartGrid_Dissertation', r'C:\Users\user\Desktop\smartrid1',
        os.path.join(home, 'Desktop', 'smartrid1', 'SmartGrid_Dissertation'), os.path.join(home, 'Desktop', 'smartrid1'), os.path.join(home, 'smartrid1')]
    RESULTS_DIRS = [d for d in dict.fromkeys(cands) if d and _has_v6(d)]
    if not RESULTS_DIRS:      # last resort: walk a few levels below cwd, home, Desktop and Documents looking for v6 files
        for root0 in dict.fromkeys([os.getcwd(), home, os.path.join(home, 'Desktop'), os.path.join(home, 'Documents')]):
            if not os.path.isdir(root0): continue
            base = root0.rstrip(os.sep).count(os.sep)
            for r_, dirs_, files_ in os.walk(root0):
                if r_.count(os.sep) - base > 4: dirs_[:] = []; continue
                dirs_[:] = [x for x in dirs_ if not x.startswith(('.', '$', 'AppData', 'node_modules', 'site-packages'))]
                if _has_v6(r_): RESULTS_DIRS.append(r_)
            if RESULTS_DIRS: break
    RESULTS_DIRS = list(dict.fromkeys(RESULTS_DIRS))
if not RESULTS_DIRS:
    raise SystemExit(f'No *_{PIPELINE_VERSION}.csv / .json files found. This kernel runs in {os.getcwd()!r} on a machine that does not hold the results. '
                     'Run NB10 on the machine where NB1-NB9 ran, or set RESULTS_DIRS in this cell to the results folder.')
OUT_DIR = RESULTS_DIRS[0]
print('reading from:', RESULTS_DIRS, '\nwriting to :', OUT_DIR)

# what each notebook is expected to have written (file-name stems, version suffix added below)
EXPECTED = {
 'NB1':  ['nb1_class_sizes', 'nb1_feature_group_ablation', 'nb1_session_audit', 'nb1_timestamp_audit'],
 'NB2':  ['optuna_xgb_trials', 'nb2_run_manifest'],
 'NB2a': ['nb2a_summary', 'nb2a_run_manifest'],
 'NB-SHAP': ['shap_group_share', 'shap_stability', 'shap_top10_per_class', 'shap_feature_subset_f1', 'nb_shap_run_manifest'],
 'NB3':  ['attack1_sweep', 'attack1_threat_model', 'nb3_run_manifest'],
 'NB3b': ['attack1_confusion_clean', 'attack1_confusion_pgd_eps01'],
 'NB3c': ['attack1w_comparison', 'attack1w_results_table', 'nb3c_run_manifest'],
 'NB3d': ['attack1x_convergence_table', 'nb3d_run_manifest'],
 'NB4':  ['attack2_evasion_table', 'attack2_paired_tests', 'nb4_run_manifest'],
 'NB4b': ['attack2b_sweep_table', 'attack2b_paired_tests', 'nb4b_run_manifest'],
 'NB4c': ['attack2c_clip_table', 'attack2c_nn_distance', 'attack2c_paired_tests', 'nb4c_run_manifest'],
 'NB4d': ['attack2d_results_table', 'attack2d_realism', 'attack2d_paired_tests', 'nb4d_run_manifest'],
 'NB4e': ['attack2e_f1_table', 'attack2e_per_class_f1', 'attack2e_per_class_recall', 'attack2e_ppo_vs_random', 'nb4e_run_manifest'],
 'NB5':  ['defence1_overall_table', 'defence1_per_class_table', 'nb5_run_manifest'],
 'NB5b': ['defence1b_overall_table', 'defence1b_per_class_table', 'defence1b_round_table', 'nb5b_run_manifest'],
 'NB5c': ['defence1c_convergence_table', 'defence1c_per_class_table', 'nb5c_run_manifest'],
 'NB6':  ['attack3_table', 'attack3_per_class', 'nb6_run_manifest'],
 'NB6b': ['attack3b_convergence_table', 'attack3b_per_class', 'nb6b_run_manifest'],
 'NB7':  ['nb7_run_manifest'],
 'NB7b': ['attack3c_tradeoff_rob_1.0', 'nb7b_run_manifest'],
 'NB15': ['figure_index', 'nb15_run_manifest'],
 'NB14': ['ablation_A_clean', 'ablation_A_recall_pc', 'ablation_A_attack', 'ablation_A_attack_pc', 'ablation_B_clean', 'ablation_B_evasion', 'ablation_B_paired', 'ablation_B_evasion_pc', 'ablation_checks', 'nb14_run_manifest'],
 'NB13': ['cx_hardware', 'cx_latency_single', 'cx_tier', 'cx_compare', 'cx_drift', 'cx_throughput', 'cx_size', 'cx_training', 'cx_checks', 'nb13_run_manifest'],
 'NB12': ['fl_main', 'fl_perclass', 'fl_compare', 'fl_curves', 'fl_physfed_weights', 'fl_dp', 'fl_attack', 'fl_comm', 'fl_partition', 'nb12_run_manifest'],
 'NB11b': ['conformalfix_coverage', 'conformalfix_perclass', 'conformalfix_compare', 'conformalfix_fold_recall', 'conformalfix_ensemble', 'nb11b_run_manifest'],
 'NB11': ['conformal_coverage', 'conformal_perclass', 'conformal_method_compare', 'conformal_attack', 'conformal_attack_recall', 'conformal_detection', 'conformal_clean_f1_ci', 'conformal_calibration', 'conformal_reliability', 'conformal_burst_structure', 'nb11_run_manifest'],
 'NB8':  ['esipcr_pcr_clean', 'esipcr_stability', 'esipcr_esi_per_class', 'esipcr_pcr_attack', 'nb8_run_manifest'],
 'NB9':  ['msu_models', 'msu_summary', 'msu_pairwise', 'msu_leakage', 'msu_ablation', 'msu_ablation_tests', 'msu_feature_sets', 'msu_feature_set_tests', 'msu_pcr', 'msu_per_marker_f1', 'nb9_run_manifest'],
}

## 1. Discover files

In [ ]:
def find_all(pattern):
    out = {}
    for d in RESULTS_DIRS:
        for p in glob.glob(os.path.join(d, pattern)): out.setdefault(os.path.basename(p), p)
    return out
CSV = find_all(f'*_{PIPELINE_VERSION}.csv'); JSN = find_all(f'*_{PIPELINE_VERSION}.json'); PNG = {**find_all(f'*_{PIPELINE_VERSION}.png'), **find_all(os.path.join(f'figures_{PIPELINE_VERSION}', f'*_{PIPELINE_VERSION}.png'))}; PKL = find_all(f'*_{PIPELINE_VERSION}.pkl')
OLD = {k: v for k, v in {**find_all('*_v4.csv'), **find_all('*_v5.csv'), **find_all('*_v4.png'), **find_all('*_v5.png')}.items()}
print(f'{len(CSV)} v6 csv | {len(JSN)} json | {len(PNG)} png | {len(PKL)} pkl | {len(OLD)} older-version csv/png files in the same folders')
rows = []
for nb, stems in EXPECTED.items():
    for st in stems:
        fn = f'{st}_{PIPELINE_VERSION}'
        have = [e for e, d in (('csv', CSV), ('json', JSN)) if f'{fn}.{e}' in d]
        rows.append(dict(notebook=nb, expected=fn, found=bool(have), kind=','.join(have)))
COMPLETE = pd.DataFrame(rows)
miss = COMPLETE[~COMPLETE.found]
summ = COMPLETE.groupby('notebook', sort=False).found.agg(['sum', 'count']).rename(columns={'sum': 'found', 'count': 'expected'})
print(summ.to_string()); print('\nMISSING expected files:'); print(miss.to_string(index=False) if len(miss) else '  none')
unlisted = sorted(set(CSV) - {f'{st}_{PIPELINE_VERSION}.csv' for s in EXPECTED.values() for st in s})
print(f'\n{len(unlisted)} v6 csv files not in the expected list (still exported to the workbook):'); print(*unlisted[:40], sep='\n')

## 2. Provenance audit (manifests)

Every notebook writes `*_run_manifest_v6.json` with its seeds, pipeline version, library versions and a UTC timestamp. All notebooks should agree on seeds and pipeline version; library versions can differ (the CUDA/xgboost build of the machine) but a change mid-pipeline is worth knowing about.

In [ ]:
man = []
for fn, p in sorted(JSN.items()):
    if 'run_manifest' not in fn: continue
    try: m = json.load(open(p))
    except Exception as e: man.append(dict(file=fn, error=str(e))); continue
    man.append(dict(file=fn, notebook=m.get('notebook'), pipeline=m.get('pipeline_version', m.get('pipeline')), seeds=str(m.get('seeds')), xgboost=m.get('xgboost'), torch=m.get('torch'), utc=m.get('utc')))
MAN = pd.DataFrame(man)
print(MAN.to_string(index=False) if len(MAN) else 'no manifests found')
issues = []
if len(MAN):
    if 'pipeline' in MAN: issues += [f'{r.file}: pipeline {r.pipeline!r} != {PIPELINE_VERSION!r}' for r in MAN.itertuples() if r.pipeline not in (PIPELINE_VERSION, None) and r.pipeline == r.pipeline]
    if 'seeds' in MAN: issues += [f'{r.file}: seeds {r.seeds} != {SEEDS}' for r in MAN.itertuples() if r.seeds not in (str(SEEDS), 'None') and r.seeds == r.seeds]
    for lib in ('xgboost', 'torch'):
        if lib in MAN and MAN[lib].dropna().nunique() > 1: issues.append(f'{lib} version differs between notebooks: {sorted(MAN[lib].dropna().unique())}')
    if 'utc' in MAN: print('\nrun order by timestamp:'); print(MAN.dropna(subset=['utc']).sort_values('utc')[['notebook', 'utc']].to_string(index=False))
print('\nPROVENANCE ISSUES:'); print(*(issues or ['none']), sep='\n')

## 3. Load all tables and run the integrity checks

Checks per table: shape, NaN and inf cells, cells written as percentages (`'1.58%'` next to fractions elsewhere), `mean +/- std` strings whose std is `0.0000` (a deterministic model or a single seed, so a paired test on that column is meaningless), and p-value-like columns containing exact zeros.

In [ ]:
TABLES = {}; audit = []
pat_pct = re.compile(r'^\s*-?\d+(\.\d+)?\s*%\s*$'); pat_pm = re.compile(r'([+-]?\d+\.?\d*)\s*(?:\+/-|±)\s*([0-9.]+)')
P_COL = re.compile(r'(^p$|^p[ _(]|[ _]p$|[ _]p[ _)]|p_raw|p_holm|pval|p-val|holm|paired)', re.I)
RATE_COL = re.compile(r'(rate|evasion|fpr|recall|precision|f1|acc)', re.I)
def add(table, sev, problem): audit.append(dict(table=table, severity=sev, problem=problem))
for fn, p in sorted(CSV.items()):
    key = fn[:-len(f'_{PIPELINE_VERSION}.csv')]
    try: df = pd.read_csv(p)
    except Exception as e: add(key, 'CHECK', f'unreadable: {e}'); continue
    TABLES[key] = df
    num = df.select_dtypes('number'); obj = df.select_dtypes('object')
    if num.size and (nn := int(num.isna().sum().sum())): add(key, 'INFO', f'{nn} NaN numeric cells (often structural: blank p for a reference row, n/a for deterministic models); confirm they are not missing results')
    if num.size and (ni := int(np.isinf(num.values.astype(float)).sum())): add(key, 'CHECK', f'{ni} inf cells')
    for c in obj.columns:
        s = obj[c].map(str)
        if s.str.match(pat_pct).any():
            sev = 'INFO' if s.nunique() <= 6 else 'CHECK'
            add(key, sev, f"percent-formatted strings in '{c}' ({'a short list of setting labels, fine' if sev == 'INFO' else 'use fractions'})")
        sat, const = False, False
        for v in s:
            m = pat_pm.search(v)
            if m and float(m.group(2)) == 0.0:
                if abs(float(m.group(1))) < 1e-3 or abs(float(m.group(1)) - 1) < 1e-3: sat = True
                else: const = True
        if const: add(key, 'CHECK', f"'{c}': mean +/- 0.0000 with a mean strictly between 0 and 1 (deterministic model, a plateau, or one seed: do not use it in a paired test)")
        elif sat: add(key, 'INFO', f"'{c}': +/- 0.0000 at a mean of 0 or 1 (saturated, expected)")
    for c in num.columns:
        if P_COL.search(str(c)) and (num[c] == 0).any():
            add(key, 'INFO', f"p-value column '{c}' prints as exactly 0 (a very small p rounded in the CSV): write '< 0.0001' in the text")
    big = [c for c in num.columns if RATE_COL.search(str(c)) and num[c].abs().max() > 1.0001]
    if big: add(key, 'CHECK', f'rate-like columns above 1 (percent vs fraction mix?): {big}')
AUDIT = pd.DataFrame(audit)
print(f'{len(TABLES)} tables loaded, {sum(len(t) for t in TABLES.values())} rows in total')
print(AUDIT.sort_values(['severity', 'table']).to_string(index=False) if len(AUDIT) else 'integrity checks: no problems found')
print('\nby severity:', AUDIT.severity.value_counts().to_dict() if len(AUDIT) else {})

## 4. Write the master workbook, Markdown dump and LaTeX tables

In [ ]:
def sheet_name(k, used):
    s = re.sub(r'[\[\]\:\*\?\/\\]', '_', k)[:31]
    while s in used: s = s[:28] + '_' + str(len(used))
    used.add(s); return s
def clean(df):
    d = df.copy()
    for c in d.select_dtypes('float').columns: d[c] = d[c].round(6)
    return d
xlsx = os.path.join(OUT_DIR, f'physar_master_results_{PIPELINE_VERSION}.xlsx'); used = {'index'}; idx_rows = []
with pd.ExcelWriter(xlsx, engine='openpyxl') as w:
    pre = []
    for k, df in TABLES.items():
        sn = sheet_name(k, used); clean(df).to_excel(w, sheet_name=sn, index=False)
        owner = next((nb for nb, st in EXPECTED.items() if any(k == s_ for s_ in st)), '')
        idx_rows.append(dict(sheet=sn, table=k, notebook=owner, rows=len(df), cols=df.shape[1]))
    IDX = pd.DataFrame(idx_rows); IDX.to_excel(w, sheet_name='index', index=False)
    COMPLETE.to_excel(w, sheet_name='completeness', index=False); MAN.to_excel(w, sheet_name='manifests', index=False)
    (AUDIT if len(AUDIT) else pd.DataFrame({'problem': ['none']})).to_excel(w, sheet_name='integrity', index=False)
    wb = w.book; wb.move_sheet('index', offset=-len(wb.sheetnames) + 1)
    for ws in wb.worksheets:
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(48, max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col[:60]) + 2))
        ws.freeze_panes = 'A2'
print('wrote', xlsx, f'({len(TABLES)} table sheets)')

def to_md(d):      # pipe table without the optional 'tabulate' dependency
    cell = lambda v: str(v).replace('|', '/').replace('\n', ' ')
    head = '| ' + ' | '.join(cell(c) for c in d.columns) + ' |\n| ' + ' | '.join('---' for _ in d.columns) + ' |\n'
    return head + ''.join('| ' + ' | '.join(cell(v) for v in r) + ' |\n' for r in d.values.tolist())
md_path = os.path.join(OUT_DIR, f'master_tables_{PIPELINE_VERSION}.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(f'# PhysAR-IDS master tables ({PIPELINE_VERSION})\n\n')
    for k, df in TABLES.items():
        f.write(f'## {k}\n\n'); f.write(to_md(clean(df))); f.write('\n\n')
print('wrote', md_path)
tex_dir = os.path.join(OUT_DIR, f'master_latex_{PIPELINE_VERSION}'); os.makedirs(tex_dir, exist_ok=True)
def esc(s): return str(s).replace('\\', r'\textbackslash{}').replace('_', r'\_').replace('%', r'\%').replace('&', r'\&').replace('#', r'\#')
for k, df in TABLES.items():
    d = clean(df).round(4)
    body = [' & '.join(esc(c) for c in d.columns) + r' \\', r'\midrule'] + [' & '.join(esc(v) for v in r) + r' \\' for r in d.values.tolist()]
    tex = '\\begin{tabular}{' + 'l' * d.shape[1] + '}\n\\toprule\n' + '\n'.join(body[:1]) + '\n' + '\n'.join(body[1:]) + '\n\\bottomrule\n\\end{tabular}\n'
    open(os.path.join(tex_dir, f'{k}.tex'), 'w', encoding='utf-8').write(tex)
print('wrote', len(TABLES), 'LaTeX tables to', tex_dir)

## 5. Figure book

Every `*_v6.png` in one PDF, one per page with the file name as the title. The dissertation's 10 stale embedded figures should each be replaced by one of these; anything in the text with no counterpart here has no v6 figure yet.

In [ ]:
pdf_path = os.path.join(OUT_DIR, f'master_figures_{PIPELINE_VERSION}.pdf'); fig_rows = []
with PdfPages(pdf_path) as pdf:
    if not PNG: fig, ax = plt.subplots(figsize=(6, 2)); ax.axis('off'); ax.text(0, 0.5, 'no v6 figures found'); pdf.savefig(fig); plt.close(fig)
    for fn, p in sorted(PNG.items()):
        try: img = plt.imread(p)
        except Exception as e: fig_rows.append(dict(figure=fn, note=f'unreadable: {e}')); continue
        h, wd = img.shape[:2]; fig, ax = plt.subplots(figsize=(11, 11 * h / wd + 0.5)); ax.imshow(img); ax.axis('off'); ax.set_title(fn, fontsize=9); pdf.savefig(fig); plt.close(fig)
        fig_rows.append(dict(figure=fn, px=f'{wd}x{h}', modified=time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))))
FIGS = pd.DataFrame(fig_rows); print(FIGS.to_string(index=False) if len(FIGS) else 'no figures found'); print('wrote', pdf_path)

## 6. Manifest and summary

In [ ]:
verdict = dict(notebooks_complete=int((summ['found'] == summ['expected']).sum()), notebooks_total=len(summ), missing_files=int((~COMPLETE.found).sum()),
               provenance_issues=len(issues), integrity_check=int((AUDIT.severity == 'CHECK').sum()) if len(AUDIT) else 0, integrity_info=int((AUDIT.severity == 'INFO').sum()) if len(AUDIT) else 0, tables=len(TABLES), figures=len(FIGS))
print(json.dumps(verdict, indent=1))
json.dump({'notebook': 'NB10', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'verdict': verdict, 'provenance_issues': issues, 'integrity': AUDIT.to_dict('records'),
           'missing': miss.expected.tolist(), 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(os.path.join(OUT_DIR, f'nb10_run_manifest_{PIPELINE_VERSION}.json'), 'w'), indent=1)
for f in [xlsx, md_path, pdf_path, os.path.join(OUT_DIR, f'nb10_run_manifest_{PIPELINE_VERSION}.json')]: print(('  OK  ' if os.path.exists(f) else 'MISSING '), os.path.basename(f))